In [3]:
import sys
!{sys.executable} -m pip install yfinance pandas

  Using cached yfinance-1.4.0-py2.py3-none-any.whl.metadata (6.2 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached peewee-4.0.6-py3-none-any.whl.metadata (8.6 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (18 kB)
  Using cached websockets-16.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached cffi-2.0.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached yfinance-1.4.0-py2.py3-none-any.whl (137 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 2.8 MB/s  0:00:03m0:00:0100:01
Using cached curl_cffi-0.15.0-cp310-abi3-macosx_11_0_arm64.whl (2.6 MB)
Using cached cffi-2.0.0-cp312-cp312-macosx_11_0_arm64.whl (181 kB)
Using c

In [5]:
import yfinance as yf
import pandas as pd

# Load the pkl file
df = pd.read_pickle('../data/raw/motley-fool-data.pkl')

print(df.head())
print(df.columns)
print(df.shape)

                         date      exchange        q ticker  \
0  Aug 27, 2020, 9:00 p.m. ET  NASDAQ: BILI  2020-Q2   BILI   
1  Jul 30, 2020, 4:30 p.m. ET     NYSE: GFF  2020-Q3    GFF   
2  Oct 23, 2019, 5:00 p.m. ET  NASDAQ: LRCX  2020-Q1   LRCX   
3  Nov 6, 2019, 12:00 p.m. ET  NASDAQ: BBSI  2019-Q3   BBSI   
4   Aug 7, 2019, 8:30 a.m. ET  NASDAQ: CSTE  2019-Q2   CSTE   

                                          transcript  
0  Prepared Remarks:\nOperator\nGood day, and wel...  
1  Prepared Remarks:\nOperator\nThank you for sta...  
2  Prepared Remarks:\nOperator\nGood day and welc...  
3  Prepared Remarks:\nOperator\nGood day, everyon...  
4  Prepared Remarks:\nOperator\nGreetings and wel...  
Index(['date', 'exchange', 'q', 'ticker', 'transcript'], dtype='object')
(18755, 5)


In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

In [10]:
df = pd.read_pickle('../data/raw/motley-fool-data.pkl')

# Fix the messy date format
df['date'] = df['date'].str.replace(r'\s+ET$', '', regex=True).str.strip()
df['date'] = pd.to_datetime(df['date'], format='%b %d, %Y, %I:%M %p', errors='coerce')

# Check how many parsed successfully
print(f"Total rows: {len(df)}")
print(f"Date parsed successfully: {df['date'].notna().sum()}")
print(f"Date parse failed: {df['date'].isna().sum()}")
print(df['date'].head(10))

Total rows: 18755
Date parsed successfully: 0
Date parse failed: 18755
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
5   NaT
6   NaT
7   NaT
8   NaT
9   NaT
Name: date, dtype: datetime64[s]


In [12]:
import pandas as pd

df = pd.read_pickle('../data/raw/motley-fool-data.pkl')

print(df.shape)
print(df.columns.tolist())
print(repr(df['date'].iloc[0]))
print(repr(df['date'].iloc[1]))
print(repr(df['date'].iloc[2]))
print(type(df['date'].iloc[0]))

(18755, 5)
['date', 'exchange', 'q', 'ticker', 'transcript']
'Aug 27, 2020, 9:00 p.m. ET'
'Jul 30, 2020, 4:30 p.m. ET'
'Oct 23, 2019, 5:00 p.m. ET'
<class 'str'>


In [13]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_pickle('../data/raw/motley-fool-data.pkl')

# Fix a.m./p.m. → AM/PM then parse
df['date'] = (df['date']
              .str.replace('a.m.', 'AM', regex=False)
              .str.replace('p.m.', 'PM', regex=False)
              .str.replace(' ET', '', regex=False)
              .str.strip())

df['date'] = pd.to_datetime(df['date'], format='%b %d, %Y, %I:%M %p', errors='coerce')

# Clean
df = df.dropna(subset=['date', 'ticker', 'transcript'])
df = df.drop_duplicates(subset=['ticker', 'q'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Total rows after cleaning: {len(df)}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(df[['date', 'ticker', 'q']].head())

Total rows after cleaning: 16803
Date range: 2019-05-07 16:30:00 → 2023-02-23 11:00:00
                 date ticker        q
0 2019-05-07 16:30:00    PEN  2019-Q1
1 2019-05-09 11:00:00    CXW  2019-Q1
2 2019-05-10 08:30:00    MDP  2019-Q3
3 2019-05-10 09:00:00    LIN  2019-Q1
4 2019-05-10 13:00:00    SUZ  2019-Q1


In [17]:
import yfinance as yf
from datetime import timedelta

def get_price_change(ticker, call_date, days_after=3):
    try:
        start = (call_date - timedelta(days=5)).strftime('%Y-%m-%d')
        end   = (call_date + timedelta(days=days_after + 5)).strftime('%Y-%m-%d')
        
        t    = yf.Ticker(ticker)
        hist = t.history(start=start, end=end)
        
        if hist.empty or len(hist) < 2:
            return None
        
        prices = hist['Close'].dropna()
        
        # Find prices on/after the call date
        call_date_only = call_date.date()
        future = prices[prices.index.date >= call_date_only]
        
        if len(future) < 2:
            return None
        
        price_before = float(future.iloc[0])
        price_after  = float(future.iloc[-1])
        pct_change   = ((price_after - price_before) / price_before) * 100
        direction    = 1 if pct_change > 0 else 0
        
        return round(pct_change, 4), direction
    except:
        return None

In [19]:
import yfinance as yf
print(yf.__version__)

# Test one known ticker
ticker = yf.Ticker("AAPL")
hist = ticker.history(period="5d")
print(hist)

1.4.0
                                 Open        High         Low       Close  \
Date                                                                        
2026-05-18 00:00:00-04:00  300.239990  300.660004  294.910004  297.839996   
2026-05-19 00:00:00-04:00  296.970001  300.510010  296.350006  298.970001   
2026-05-20 00:00:00-04:00  298.179993  302.799988  298.079987  302.250000   
2026-05-21 00:00:00-04:00  301.059998  305.540009  300.399994  304.989990   
2026-05-22 00:00:00-04:00  306.119995  311.399994  305.839996  308.820007   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-05-18 00:00:00-04:00  34483000        0.0           0.0  
2026-05-19 00:00:00-04:00  42243600        0.0           0.0  
2026-05-20 00:00:00-04:00  38229800        0.0           0.0  
2026-05-21 00:00:00-04:00  42965100        0.0           0.0  
2026-05-22 00:00:00-04:00  43627900        0.0           0.0  


In [18]:
import time

sample_df = df.sample(200, random_state=42).copy()

results = []
failed  = 0

for i, row in sample_df.iterrows():
    result = get_price_change(row['ticker'], row['date'])
    if result:
        results.append({
            'ticker':     row['ticker'],
            'date':       row['date'],
            'pct_change': result[0],
            'direction':  result[1]
        })
    else:
        failed += 1
    
    # Small delay to avoid rate limiting
    time.sleep(0.1)
    
    if (len(results) + failed) % 50 == 0:
        print(f"✓ {len(results)} fetched | ✗ {failed} failed")

print(f"\nDone — {len(results)} success, {failed} failed")
price_df = pd.DataFrame(results)
price_df.head()

$DVAX: possibly delisted; no timezone found
$KRA: possibly delisted; no timezone found
$SHYF: possibly delisted; no timezone found
$TGP: possibly delisted; no timezone found
$LHDX: possibly delisted; no timezone found
$BIG: possibly delisted; no timezone found


✓ 44 fetched | ✗ 6 failed


$INT: possibly delisted; no timezone found
$SPPI: possibly delisted; no timezone found
$DESP: possibly delisted; no timezone found
$SNV: possibly delisted; no timezone found
$WRE: possibly delisted; no timezone found
$GPS: possibly delisted; no timezone found
$CONN: possibly delisted; no timezone found
$WLTW: possibly delisted; no timezone found


✓ 86 fetched | ✗ 14 failed


$REGI: possibly delisted; no timezone found
$EB: possibly delisted; no timezone found
$OSH: possibly delisted; no timezone found
$DOOO: possibly delisted; no timezone found
$NBEV: possibly delisted; no timezone found
$ISEE: possibly delisted; no timezone found
$HUD: possibly delisted; no timezone found
$ATVI: possibly delisted; no timezone found
$HOUS: possibly delisted; no timezone found
$RADA: possibly delisted; no timezone found
$AZPN: possibly delisted; no timezone found
$ALBO: possibly delisted; no timezone found
$LTHM: possibly delisted; no timezone found
$SIVB: possibly delisted; no timezone found
$OPI: possibly delisted; no timezone found
$CRY: possibly delisted; no price data found  (1d 2020-02-08 -> 2020-02-21) (Yahoo error = "Data doesn't exist for startDate = 1581138000, endDate = 1582261200")
$TWOU: possibly delisted; no timezone found


✓ 119 fetched | ✗ 31 failed


$K: possibly delisted; no timezone found
$EQC: possibly delisted; no timezone found
$BKCC: possibly delisted; no timezone found
$LSI: possibly delisted; no timezone found
$MPW: possibly delisted; no timezone found
$SAND: possibly delisted; no timezone found
$KIRK: possibly delisted; no timezone found
$ZI: possibly delisted; no timezone found


✓ 161 fetched | ✗ 39 failed

Done — 161 success, 39 failed


,ticker,date,pct_change,direction
0,RELY,2022-03-02 17:00:00,-12.4771,0
1,CNF,2020-11-24 20:00:00,4.7904,1
2,BUD,2021-10-28 09:00:00,-4.8457,0
3,AFIB,2022-03-30 16:30:00,-39.8964,0
4,MNDY,2022-02-23 08:30:00,24.0056,1


In [20]:
merged = sample_df.merge(price_df, on=['ticker', 'date'], how='inner')

print(f"Merged shape: {merged.shape}")
print(f"Up (1):  {merged['direction'].sum()} | Down (0): {(merged['direction']==0).sum()}")
print(f"Up %:    {merged['direction'].mean()*100:.1f}%")
print(f"Avg % change: {merged['pct_change'].mean():.2f}%")
print(f"Max gain: {merged['pct_change'].max():.2f}% | Max drop: {merged['pct_change'].min():.2f}%")

merged.to_csv('../data/processed/transcripts_with_prices.csv', index=False)
print("\n✅ Saved to data/processed/transcripts_with_prices.csv")

merged[['ticker', 'date', 'q', 'pct_change', 'direction']].head(10)

Merged shape: (161, 7)
Up (1):  70 | Down (0): 91
Up %:    43.5%
Avg % change: -0.70%
Max gain: 30.32% | Max drop: -39.90%

✅ Saved to data/processed/transcripts_with_prices.csv


,ticker,date,q,pct_change,direction
0,RELY,2022-03-02 17:00:00,2021-Q4,-12.4771,0
1,CNF,2020-11-24 20:00:00,2020-Q3,4.7904,1
2,BUD,2021-10-28 09:00:00,2021-Q3,-4.8457,0
3,AFIB,2022-03-30 16:30:00,2021-Q4,-39.8964,0
4,MNDY,2022-02-23 08:30:00,2021-Q4,24.0056,1
5,SPOT,2021-07-28 08:00:00,2021-Q2,-2.6420,0
6,TITN,2021-11-23 08:30:00,2022-Q3,-13.1888,0
7,ENR,2021-11-10 10:00:00,2021-Q4,-1.1699,0
8,SMTC,2019-12-04 17:00:00,2020-Q3,1.3037,1
9,IBEX,2022-09-22 16:30:00,2022-Q4,16.4622,1


In [21]:
import time

results_full = []
failed_full  = 0

for i, row in df.iterrows():
    result = get_price_change(row['ticker'], row['date'])
    if result:
        results_full.append({
            'ticker':     row['ticker'],
            'date':       row['date'],
            'q':          row['q'],
            'transcript': row['transcript'],
            'pct_change': result[0],
            'direction':  result[1]
        })
    else:
        failed_full += 1

    time.sleep(0.05)

    # Auto-save every 1000 rows so you don't lose progress
    if len(results_full) % 1000 == 0 and len(results_full) > 0:
        pd.DataFrame(results_full).to_pickle('../data/processed/full_dataset_partial.pkl')
        print(f"✓ {len(results_full)} saved | ✗ {failed_full} failed")

# Final save
full_df = pd.DataFrame(results_full)
full_df.to_pickle('../data/processed/full_dataset.pkl')
print(f"\n✅ Complete — {len(full_df)} rows saved to full_dataset.pkl")

$MDP: possibly delisted; no timezone found
$IGT: possibly delisted; no timezone found
$HIBB: possibly delisted; no timezone found
$HMLP: possibly delisted; no timezone found
$EXPR: possibly delisted; no timezone found
$VMW: possibly delisted; no timezone found
$BIG: possibly delisted; no timezone found
$CTK: possibly delisted; no timezone found
$COUP: possibly delisted; no timezone found
$TCS: possibly delisted; no timezone found
$SKX: possibly delisted; no timezone found
$KSU: possibly delisted; no timezone found
$IPG: possibly delisted; no timezone found
$ACC: possibly delisted; no timezone found
$IBA: possibly delisted; no timezone found
$PSB: possibly delisted; no timezone found
$XLNX: possibly delisted; no timezone found
$TPX: possibly delisted; no timezone found
$AXE: possibly delisted; no timezone found
$VVI: possibly delisted; no timezone found
$CXP: possibly delisted; no timezone found
$VNE: possibly delisted; no timezone found
$ABTX: possibly delisted; no timezone found
$WRE:

✓ 1000 saved | ✗ 234 failed


$UNVR: possibly delisted; no timezone found
$SRC: possibly delisted; no timezone found
$PGTI: possibly delisted; no timezone found
$SYX: possibly delisted; no timezone found
$ODP: possibly delisted; no timezone found
$FRGI: possibly delisted; no timezone found
$VRS: possibly delisted; no timezone found
$ETRN: possibly delisted; no timezone found
$BIG: possibly delisted; no timezone found
$VMW: possibly delisted; no timezone found
$INT: possibly delisted; no timezone found
$EB: possibly delisted; no timezone found
$TRHC: possibly delisted; no timezone found
$TPRE: possibly delisted; no timezone found
$DRQ: possibly delisted; no timezone found
$HHC: possibly delisted; no timezone found
$ROCC: possibly delisted; no timezone found
$BCEI: possibly delisted; no timezone found
$CIR: possibly delisted; no timezone found
$AMBC: possibly delisted; no timezone found
$CVET: possibly delisted; no timezone found
$ATTO: possibly delisted; no timezone found
$AXNX: possibly delisted; no timezone found


✓ 2000 saved | ✗ 529 failed


$ROLL: possibly delisted; no timezone found


...

✓ 12000 saved | ✗ 3175 failed


$HMST: possibly delisted; no timezone found
$SKX: possibly delisted; no timezone found
$HA: possibly delisted; no timezone found
$CS: possibly delisted; no timezone found
$ROIC: possibly delisted; no timezone found
$CHX: possibly delisted; no timezone found
$COOP: possibly delisted; no timezone found
$PFC: possibly delisted; no timezone found
$NCR: possibly delisted; no timezone found
$MYOV: possibly delisted; no timezone found
$OSTK: possibly delisted; no timezone found
$CBD: possibly delisted; no timezone found
$HEES: possibly delisted; no timezone found
$SJW: possibly delisted; no timezone found
$AIMC: possibly delisted; no timezone found
$SGEN: possibly delisted; no timezone found
$NATI: possibly delisted; no timezone found
$INFN: possibly delisted; no timezone found
$IMGN: possibly delisted; no timezone found
$X: possibly delisted; no timezone found
$AUY: possibly delisted; no timezone found
$ZI: possibly delisted; no timezone found
$IGT: possibly delisted; no timezone found
$SPWR

✓ 13000 saved | ✗ 3377 failed


$TWNK: possibly delisted; no timezone found
$CFLT: possibly delisted; no timezone found
$CDAY: possibly delisted; no timezone found
$INFN: possibly delisted; no timezone found
$PYCR: possibly delisted; no price data found  (1d 2022-10-28 -> 2022-11-10)
$ISEE: possibly delisted; no timezone found
$NGD: possibly delisted; no timezone found
$NKLA: possibly delisted; no timezone found
$ORCC: possibly delisted; no timezone found
$GLT: possibly delisted; no timezone found
$DVAX: possibly delisted; no timezone found
$EXAS: possibly delisted; no timezone found
$SQ: possibly delisted; no timezone found
$MODG: possibly delisted; no timezone found
$HEAR: possibly delisted; no timezone found
$SWAV: possibly delisted; no timezone found
$SDC: possibly delisted; no timezone found
$IGT: possibly delisted; no timezone found
$SPWR: possibly delisted; no price data found  (1d 2022-11-03 -> 2022-11-16) (Yahoo error = "Data doesn't exist for startDate = 1667448000, endDate = 1668574800")
$NVTA: possibly de


✅ Complete — 13377 rows saved to full_dataset.pkl
